# Wafer Map of Downstream Status

This notebook loads the saved CSV files and visualizes three downstream states on a wafer map:

- `Pass`
- `Fail`
- `Not tested / no valid downstream record`

The third category combines dies that were never sampled downstream and dies that were sampled but have `test_valid = 0`.

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.utils import load_sources

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 8)

In [ ]:
df_inline, df_downstream = load_sources(input_dir='../data', prefix='synthetic')

print(f'Inline rows: {len(df_inline)}')
print(f'Downstream sampled rows: {len(df_downstream)}')
print(f'Valid downstream rows: {(df_downstream['test_valid'] == 1).sum()}')
print(f'Invalid downstream rows: {(df_downstream['test_valid'] == 0).sum()}')

In [ ]:
df_status = df_inline.merge(
    df_downstream[['wafer_id', 'die_id', 'test_pass', 'test_valid']],
    on=['wafer_id', 'die_id'],
    how='left',
)

def classify_status(row):
    if pd.isna(row['test_valid']) or row['test_valid'] == 0:
        return 'Not tested / no valid downstream record'
    if row['test_pass'] == 1:
        return 'Pass'
    return 'Fail'

df_status['downstream_status'] = df_status.apply(classify_status, axis=1)
df_status['downstream_status'] = pd.Categorical(
    df_status['downstream_status'],
    categories=['Pass', 'Fail', 'Not tested / no valid downstream record'],
    ordered=True,
)

df_status[['wafer_id', 'die_id', 'downstream_status']].head()

In [ ]:
wafer_summary = (
    df_status.groupby(['wafer_id', 'downstream_status'], observed=False)
    .size()
    .unstack(fill_value=0)
    .sort_values(['Fail', 'Not tested / no valid downstream record'], ascending=False)
)

wafer_summary.head()

In [ ]:
selected_wafer = wafer_summary.index[0]
print(f'Selected wafer: {selected_wafer}')
wafer_summary.loc[selected_wafer]

In [ ]:
palette = {
    'Pass': '#2C7FB8',
    'Fail': '#D95F0E',
    'Not tested / no valid downstream record': '#BDBDBD',
}

wafer_df = df_status[df_status['wafer_id'] == selected_wafer].copy()

fig, ax = plt.subplots(figsize=(9, 9))
for status, color in palette.items():
    subset = wafer_df[wafer_df['downstream_status'] == status]
    ax.scatter(
        subset['x_mm'],
        subset['y_mm'],
        s=36,
        alpha=0.85,
        color=color,
        edgecolors='none',
        label=f'{status} ({len(subset)})',
    )

ax.set_title(f'Downstream Status Wafer Map: {selected_wafer}')
ax.set_xlabel('x_mm')
ax.set_ylabel('y_mm')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='upper right', frameon=True)
ax.grid(True, alpha=0.25)
plt.show()